```mermaid
flowchart LR
    A0["00"] --> A1a["01a"] --> A1b["01b"] --> A2["02"] --> A3["03"] --> A4a["04a"] --> A4b["04b"]
    A4b --> A5a["05a"] --> A5b["05b"] --> A6a["06a"] --> A6b["06b"]
    A6b --> A7["07"] --> A8a["08a"] --> A8b["08b"]
    A8b --> A9["09"] --> A10["10"] --> A11["11"] --> A12["12"] 
    
    classDef normal fill:#f8f9fa,stroke:#adb5bd,stroke-width:1px,color:#111;
    classDef done fill:#e8f7f0,stroke:#198754,stroke-width:1.5px,color:#111;
    classDef current fill:#fff3cd,stroke:#ff8c00,stroke-width:2px,color:#111;
    
    class A0,A1a,A1b,A2,A3,A4a,A4b,A5a,A5b,A6a,A6b,A7,A8a,A8b done;
    class A9 current;
    class A10,A11,A12 normal;
```

# Notebook 09 — Text Classification: Training, Evaluation, and Error Analysis

This notebook introduces supervised text classification as a complete methodological workflow: from feature matrices and labels to train/test splits, evaluation, interpretation, and error analysis.

We build on the feature representations created in Notebook 07 (Bag-of-Words and TF–IDF) and optionally compare them with embedding-based features from Notebook 08.

The guiding methodological question is:

> How do representation choices shape what a classifier learns — and how can we evaluate whether the model is learning meaningful patterns rather than artifacts?

This notebook emphasizes:
- careful train/validation/test design
- avoiding data leakage
- interpretable baselines
- evaluation beyond accuracy
- inspection of feature weights and errors
- responsible interpretation of classification outputs

## Learning goals

By the end of this notebook, students should be able to:

- create supervised train/test splits for text classification
- explain why leakage invalidates evaluation
- train baseline linear text classifiers
- compare Bag-of-Words and TF–IDF features
- evaluate models using accuracy, precision, recall, and macro/micro F1
- interpret confusion matrices
- inspect influential features in linear models
- perform qualitative error analysis
- reflect critically on what a classifier may actually be learning

## Method note

In NLP, strong performance metrics do not automatically imply meaningful understanding. A classifier may learn:

- historical vocabulary differences
- OCR artifacts
- stylistic conventions
- author-specific language
- metadata leakage

instead of the conceptual distinctions we intended to model.

For this reason, this notebook emphasizes *evaluation-by-inspection*: we inspect confusion matrices, feature weights, and misclassified examples rather than relying on a single metric. We will also add a cross-validation step and evaluate one of the parameters that we use in order to explore how parameters can impact the results and hence the interpretation. 

In [ ]:
from __future__ import annotations

from pathlib import Path
import json
import re

import numpy as np
import pandas as pd

from scipy import sparse

from sklearn.model_selection import train_test_split
from sklearn.pipeline import Pipeline
from sklearn.linear_model import LogisticRegression
from sklearn.naive_bayes import MultinomialNB
from sklearn.metrics import (
    accuracy_score,
    precision_recall_fscore_support,
    classification_report,
    confusion_matrix,
    ConfusionMatrixDisplay,
)
from sklearn.feature_extraction.text import (
    CountVectorizer,
    TfidfVectorizer,
    ENGLISH_STOP_WORDS,
)

import matplotlib.pyplot as plt
import seaborn as sns

import spacy
from spacy.tokens import DocBin

from IPython.display import display

In [ ]:
# ------------------------------------------------------------
# PATHS
# ------------------------------------------------------------
PROJECT_ROOT = Path('.')

DATA_DIR = PROJECT_ROOT / 'data'
PROCESSED_DIR = DATA_DIR / 'processed'

ANALYSIS_DIR = PROJECT_ROOT / 'analysis'
FIGURES_DIR = ANALYSIS_DIR / 'figures'
TABLES_DIR = ANALYSIS_DIR / 'tables'
REPORTS_DIR = ANALYSIS_DIR / 'reports'
MODELS_DIR = ANALYSIS_DIR / 'models'
CACHE_DIR = PROJECT_ROOT / 'cache'

for p in [FIGURES_DIR, TABLES_DIR, REPORTS_DIR, MODELS_DIR, CACHE_DIR]:
    p.mkdir(parents=True, exist_ok=True)

DOC_INDEX = TABLES_DIR / 'nb03-doc_index.csv'
SPLIT_DIR = PROCESSED_DIR / 'nb05-corpus-split'

print('DOC_INDEX:', DOC_INDEX)
print('SPLIT_DIR:', SPLIT_DIR)

## Load the canonical document table

We load the stable document index generated in notebook 03. This provides consistent identifiers, filenames, and time bins across the workflow.

# Fill the gap

In [ ]:
# =============================================== YOUR CODE HERE ===============================================
df = # Use pandas to load the CSV that contains the corpus metadata generated in notebook 03
df["publication_year"] = pd.to_numeric(df["publication_year"], errors="coerce").astype("Int64")

print('Documents:', len(df))
display(df.head())

## Load the split spaCy corpus

Notebook 05a serialized the annotated corpus into split `DocBin` files.

Why does this matter?
- we avoid reparsing the entire corpus
- annotation is expensive
- later notebooks focus on modeling rather than preprocessing

# Fill the gap

In [ ]:
spacy_files = sorted(SPLIT_DIR.glob('*.spacy'))
print(f'Found {len(spacy_files)} split files.')

# =============================================== YOUR CODE HERE ===============================================
nlp =                                # Load spacy's lightweight pipeline vocabulary and disable the NER pipeline

docs = []
for fp in spacy_files:
    db = DocBin().from_disk(fp)
    docs.extend(list(db.get_docs(nlp.vocab)))

print('Loaded docs:', len(docs))

## Reconstruct document texts

Classification models require raw text strings as inputs. We reconstruct them from the serialized spaCy documents.

In [ ]:
chunk_rows = []
for doc in docs:
    # Retrieve metadata from the Doc
    pg_id = doc.user_data.get('pg_id')
    title = doc.user_data.get('title') 
    pub_year = doc.user_data.get('publication_year')
    time_bin = doc.user_data.get('time_bin')
    chunk_index = doc.user_data.get('chunk_index', 0)
    
    chunk_rows.append({
        'pg_id': pg_id,
        'title': title,
        'publication_year': pub_year,
        'time_bin': time_bin,
        'chunk_index': chunk_index,
        'text': doc.text,          # the actual text of this chunk
        'n_tokens': len(doc)
    })

chunk_df = pd.DataFrame(chunk_rows)
print(f"\nCreated chunk DataFrame with {len(chunk_df)} rows.")
display(chunk_df.head())

# Choosing a classification target

In supervised learning, the target variable defines what the model is trying to predict.

For this notebook we use `time_bin` as the prediction target.

This creates an important interpretive question:

> If the model predicts historical period successfully, what linguistic signals is it actually using?

Possible answers include:
- historical vocabulary shifts
- genre conventions
- author-specific style
- OCR artifacts
- genuinely changing concepts

Part of the notebook's goal is to investigate these possibilities critically.

In [ ]:
# Classification target
TARGET_COL = 'time_bin'

In [ ]:
df_model = chunk_df.dropna(subset=[TARGET_COL]).copy()

print('Documents with labels:', len(df_model))
print(df_model[TARGET_COL].value_counts().sort_index())

# Train/test splits and leakage

A core methodological issue in machine learning is avoiding *data leakage*.

Leakage occurs when information from the test set influences training, directly or indirectly. This produces unrealistically optimistic evaluation scores.

Examples of leakage in NLP:
- fitting TF–IDF on the full corpus before splitting
- duplicate documents across train/test
- metadata fields accidentally included in features
- splitting chunks from the same document across train/test

We therefore split the data first, then fit vectorizers only on the training data.

In [ ]:
# Split settings
TEST_SIZE = 0.2
RANDOM_STATE = 42

In [ ]:
X_text = df_model['text'].astype(str)
y = df_model[TARGET_COL].astype(str)

X_train, X_test, y_train, y_test = train_test_split(
    X_text,
    y,
    test_size=TEST_SIZE,
    random_state=RANDOM_STATE,
    stratify=y,
)

print('Train size:', len(X_train))
print('Test size:', len(X_test))

## Cross-validation and feature-design sensitivity

Before fitting a final model on the full training set, we briefly use cross-validation to examine two methodological issues: how much performance varies across folds, and how sensitive results are to a simple TF–IDF design choice such as `min_df`.

So far, we have evaluated models with a single train/test split. This is useful, but it can make performance look more stable than it really is. In practice, model scores vary depending on how the data is split, and they may also change when we adjust the feature representation.

To make this more visible, we run a small **5-fold cross-validation** experiment and vary one TF–IDF hyperparameter: `min_df`, the minimum number of documents in which a term must appear before it is kept. This lets us see both the **variance of evaluation scores across folds** and the **sensitivity of results to a feature-design choice** that Notebook 07 introduced conceptually.

This cross-validation experiment is run only on the training split. The held-out test set remains untouched until the final evaluation stage.

In [ ]:
# ------------------------------------------------------------
# Imports
# ------------------------------------------------------------
from sklearn.model_selection import StratifiedKFold, cross_validate
from sklearn.pipeline import Pipeline
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.linear_model import LogisticRegression

# ------------------------------------------------------------
# Experiment settings
# ------------------------------------------------------------
MIN_DF_VALUES = [2, 5, 10]
# The values of min_df we want to compare. min_df controls how rare a term
# can be before it's dropped from the vocabulary (e.g. min_df=5 means a term
# must appear in at least 5 chunks to be kept). We're treating this as a
# hyperparameter here: rather than picking one value by hand, we test a few
# and let cross-validation tell us which works best for classification.

CV_FOLDS = 5
# Number of folds for cross-validation. Instead of a single train/test split
# (which can be noisy — you might get a "lucky" or "unlucky" split), we split
# the training data into 5 parts, train on 4 and evaluate on the 5th, and
# rotate which part is held out. This gives 5 performance estimates per
# min_df value instead of just one, so we can see both the average score
# and how much it varies.

# Feature settings (min_df is swept via MIN_DF_VALUES above, not fixed here)
MAX_FEATURES = 50000
NGRAM_RANGE = (1, 2)
# Kept fixed across all runs so that min_df is the only thing that changes
# between configurations — this isolates its effect on performance rather
# than mixing it with other confounding changes.

# ------------------------------------------------------------
# Set up cross-validation splitter
# ------------------------------------------------------------
cv = StratifiedKFold(n_splits=CV_FOLDS, shuffle=True, random_state=RANDOM_STATE)
# "Stratified" means each fold preserves the same class proportions as the
# full dataset — important for classification, especially if some classes
# are rarer than others. shuffle=True randomizes which rows go into which
# fold; random_state makes that shuffle reproducible.

rows = []  # will collect one summary row per min_df value tested

# ------------------------------------------------------------
# Try each min_df value and cross-validate a full pipeline for each
# ------------------------------------------------------------
for min_df in tqdm(MIN_DF_VALUES, desc="Cross-validating min_df values"):

    # A Pipeline chains preprocessing and modeling into a single object.
    # This matters for cross-validation specifically: if we vectorized the
    # full dataset once and then split into folds, information from the
    # validation fold would leak into the vocabulary used to train on the
    # other folds. Wrapping both steps in a Pipeline ensures the vectorizer
    # is re-fit from scratch on only the training portion of each fold.
    pipe = Pipeline([
        ("tfidf", TfidfVectorizer(
            lowercase=True,
            min_df=min_df,              # <- the value being swept this iteration
            max_features=MAX_FEATURES,
            ngram_range=NGRAM_RANGE,
        )),
        ("clf", LogisticRegression(
            max_iter=1000,              # allow more optimization steps to reach convergence
            random_state=RANDOM_STATE,  # reproducible results across runs
        )),
    ])

    # cross_validate handles the fold-by-fold train/evaluate loop for us:
    # for each of the 5 folds, it fits `pipe` on the training folds and
    # scores it on the held-out fold, using the metrics listed below.
    scores = cross_validate(
        pipe,
        X_train,
        y_train,
        cv=cv,
        scoring={
            "accuracy": "accuracy",     # fraction of correct predictions
            "macro_f1": "f1_macro",     # F1 averaged equally across classes —
                                         # more informative than accuracy when
                                         # classes are imbalanced, since it
                                         # doesn't let a large class dominate
        },
        n_jobs=-1,                      # run folds in parallel across all CPU cores
        return_train_score=False,       # we only need held-out (test) scores here
    )

    # scores["test_accuracy"] and scores["test_macro_f1"] are each arrays of
    # length CV_FOLDS (one score per fold). We summarize with mean (typical
    # performance) and std (how much performance varies fold-to-fold — a
    # rough indicator of stability, not just quality).
    rows.append({
        "min_df": min_df,
        "cv_accuracy_mean": scores["test_accuracy"].mean(),
        "cv_accuracy_std": scores["test_accuracy"].std(),
        "cv_macro_f1_mean": scores["test_macro_f1"].mean(),
        "cv_macro_f1_std": scores["test_macro_f1"].std(),
    })

# ------------------------------------------------------------
# Collect and display results
# ------------------------------------------------------------
cv_results = pd.DataFrame(rows).sort_values("min_df")
display(cv_results)
# Compare cv_accuracy_mean / cv_macro_f1_mean across rows to see whether a
# stricter min_df (dropping more rare terms) helps or hurts classification —
# and check the _std columns to see whether any gains are consistent across
# folds or just noise.

In [ ]:
plt.figure(figsize=(8, 5))
plt.errorbar(
    cv_results["min_df"],
    cv_results["cv_macro_f1_mean"],
    yerr=cv_results["cv_macro_f1_std"],
    marker="o",
    capsize=4,
)
plt.title("5-fold CV macro F1 across TF-IDF min_df settings")
plt.xlabel("min_df")
plt.ylabel("Macro F1 (mean ± std)")
plt.grid(True, alpha=0.3)
plt.show()

## Interpretation

A useful takeaway is that model quality is not a single fixed number. It varies across data splits, and it can improve or worsen when we change apparently simple representation choices such as `min_df`. This is one reason why feature engineering should be treated as part of the methodological argument, not just as technical preprocessing.

# Baseline model: TF–IDF + Logistic Regression

The cross-validation sweep above compared several `min_df` values using only the training set, to get a fair sense of how each setting generalizes before touching the test set. Here we lock in a single configuration — `MIN_DF = 3`, `MAX_FEATURES = 50000`, `NGRAM_RANGE = (1, 2)` — and train one final pipeline on the full training set.

Unlike the sweep, this cell doesn't cross-validate: it fits once on `X_train`, then predicts on `X_test`, which we've held out untouched until now. This gives us actual predictions (`y_pred`) to evaluate against the true test labels in the next step; accuracy alone will not tell the full story, hence we will look out for a fuller breakdown (e.g. per-class precision/recall, a confusion matrix) next.

We begin with simple, interpretable linear baselines:

- Multinomial Naive Bayes
- Logistic Regression

Why start simple?
- fast and CPU-friendly
- easier to interpret
- strong baselines for many NLP tasks
- reveal what features the model relies on

In [ ]:
STOP_WORDS_FILE = Path('./analysis/stop_words_custom.txt')

def load_stopwords(filepath:Path = STOP_WORDS_FILE) -> set:
    """
    Load custom stop words from a plain text file (one per line).
    Lines starting with '#' are ignored (comments).
    """
    if not filepath.exists():
        raise FileNotFoundError(f"Stopword file not found: {filepath}")
    
    stopwords = set()
    with open(filepath, 'r', encoding='utf-8') as f:
        for line in f:
            line = line.strip()
            if line and not line.startswith('#'):
                stopwords.add(line.lower())
    return stopwords

CUSTOM_STOPWORDS = load_stopwords()
print(f"\nLoaded {len(CUSTOM_STOPWORDS)} custom stop words.")

STOPWORDS = list(set(ENGLISH_STOP_WORDS) | CUSTOM_STOPWORDS)
print(f"\nStored {len(STOPWORDS)} stop words in `STOPWORDS`.")


## TF–IDF + Logistic Regression pipeline

In [ ]:
# ------------------------------------------------------------
# Feature settings
# ------------------------------------------------------------
MIN_DF = 3
# Ignore terms that appear in fewer than 3 chunks — filters out rare,
# idiosyncratic terms unlikely to generalize to unseen text.

MAX_FEATURES = 50000
# Cap the vocabulary at the 50,000 most frequent terms, keeping the
# feature matrix a manageable size.

NGRAM_RANGE = (1, 2)
# Unigrams + bigrams, so short two-word phrases ("natural law") are
# captured as single features alongside individual words.

# ------------------------------------------------------------
# Build the pipeline: TF-IDF vectorizer + Logistic Regression classifier
# ------------------------------------------------------------
# As before, wrapping both steps in a Pipeline means the vectorizer's
# vocabulary and IDF weights are learned only from the training data
# (via .fit below) — the test set stays completely unseen until prediction.
tfidf_lr = Pipeline([
    ('tfidf', TfidfVectorizer(
        lowercase=True,
        stop_words=STOPWORDS,     # remove common function words using our
                                   # project's stopword list before counting
        min_df=MIN_DF,
        max_features=MAX_FEATURES,
        ngram_range=NGRAM_RANGE,
    )),
    ('clf', LogisticRegression(
        max_iter=2000,            # more optimization steps than the default,
                                   # giving the solver room to converge on a
                                   # larger TF-IDF feature space
        random_state=RANDOM_STATE,  # reproducible results across runs
    ))
])

# ------------------------------------------------------------
# Fit on the training set, then predict on the held-out test set
# ------------------------------------------------------------
print('Training TF–IDF + Logistic Regression...')
tfidf_lr.fit(X_train, y_train)
# .fit() runs both pipeline steps in sequence: first the vectorizer learns
# its vocabulary/IDF weights from X_train and transforms it into a TF-IDF
# matrix, then the classifier is trained on that matrix against y_train.

y_pred = tfidf_lr.predict(X_test)
# .predict() reuses the vectorizer's vocabulary and IDF weights learned
# from training (X_test is only transformed, never used to refit the
# vectorizer), then applies the trained classifier to produce predictions.

print('Training completed.')
# y_pred now holds one predicted label per row in X_test — ready to be
# compared against y_test (e.g. with classification_report or a confusion
# matrix) in the next cell to see how well the model actually performs.

# Evaluation metrics

Accuracy alone can be misleading, especially when classes are imbalanced.

We therefore report:
- accuracy
- precision
- recall
- macro F1
- micro F1

Important distinction:
- **Macro F1** treats all classes equally
- **Micro F1** weights larger classes more heavily

Reflection question:

> Why might macro F1 be more informative for uneven historical time bins?

In [ ]:
acc = accuracy_score(y_test, y_pred)

prec_macro, rec_macro, f1_macro, _ = precision_recall_fscore_support(
    y_test, y_pred, average='macro', zero_division=0
)

prec_micro, rec_micro, f1_micro, _ = precision_recall_fscore_support(
    y_test, y_pred, average='micro', zero_division=0
)

metrics_df = pd.DataFrame({
    'metric': [
        'accuracy',
        'precision_macro',
        'recall_macro',
        'f1_macro',
        'precision_micro',
        'recall_micro',
        'f1_micro',
    ],
    'value': [
        acc,
        prec_macro,
        rec_macro,
        f1_macro,
        prec_micro,
        rec_micro,
        f1_micro,
    ]
})

display(metrics_df)

## Classification report

In [ ]:
print(classification_report(y_test, y_pred, zero_division=0))

# Confusion matrix

Confusion matrices reveal *which classes are confused with which others*.

This is often more informative than a single summary metric.

Questions to consider:
- Which historical periods are difficult to distinguish?
- Are adjacent periods more confusable than distant ones?
- What kinds of vocabulary might explain these confusions?

In [ ]:
def bin_start(label) -> int:
    s = str(label)
    m = re.search(r'-?\d+', s.replace('–', '-'))
    return int(m.group(0)) if m else 10**9

def sort_time_bins(values) -> list[str]:
    vals = [str(v) for v in values if pd.notna(v)]
    return sorted(set(vals), key=bin_start) 

In [ ]:
# Get the unique labels, sorted chronologically using your custom function
ordered_labels = sort_time_bins(y.unique())

# Build the confusion matrix with the ordered labels
cm = confusion_matrix(y_test, y_pred, labels=ordered_labels)

# Plot it
fig, ax = plt.subplots(figsize=(10, 8))

ConfusionMatrixDisplay(
    confusion_matrix=cm,
    display_labels=ordered_labels
).plot(ax=ax, xticks_rotation=45, cmap='Blues')

plt.title('Confusion matrix: TF–IDF + Logistic Regression')
plt.tight_layout()
plt.show()

# Inspecting feature weights

Linear classifiers provide interpretable feature weights.

For each class (time bin), we can inspect the words and phrases with the strongest positive coefficients.

Important methodological warning:

> High-weight terms are not necessarily philosophically meaningful. They may reflect:
> - OCR artifacts
> - archaic spelling
> - author names
> - formatting conventions
> - genuine conceptual differences

Interpreting weights therefore requires qualitative inspection.

In [ ]:
vec = tfidf_lr.named_steps['tfidf']
clf = tfidf_lr.named_steps['clf']

feature_names = np.array(vec.get_feature_names_out())
classes = clf.classes_

rows = []

for i, c in enumerate(classes):
    top_idx = np.argsort(clf.coef_[i])[-20:][::-1]

    for j in top_idx:
        rows.append({
            'class': c,
            'feature': feature_names[j],
            'weight': float(clf.coef_[i, j]),
        })

weights_df = pd.DataFrame(rows)

display(weights_df.head(40))

# Error analysis

Error analysis is one of the most important parts of NLP modeling.

We inspect:
- confidently wrong predictions
- recurring confusion patterns
- possible annotation or metadata problems
- examples where the model appears to rely on superficial cues

Reflection question:

> What kinds of mistakes might still be acceptable for exploratory historical analysis?

In [ ]:
errors = pd.DataFrame({
    'text': X_test.tolist(),
    'true_label': y_test.tolist(),
    'pred_label': y_pred.tolist(),
})

errors = errors[errors['true_label'] != errors['pred_label']].copy()

print('Misclassified examples:', len(errors))

def snippet(text: str, n: int = 250) -> str:
    text = re.sub(r'\s+', ' ', str(text)).strip()
    return text[:n] + ('…' if len(text) > n else '')

errors['snippet'] = errors['text'].map(snippet)

display(errors[['true_label', 'pred_label', 'snippet']].head(20))

# Comparing feature representations

We now compare a second representation: Bag-of-Words counts with Naive Bayes.

The goal is not necessarily to achieve higher accuracy, but to understand how representation choices affect:
- model behavior
- interpretability
- robustness
- confusion patterns

# Fill the gap

Complete the code below where you see `# Complete`

In [ ]:
# ------------------------------------------------------------
# Second model: Bag-of-Words + Multinomial Naive Bayes
# ------------------------------------------------------------
# Reuses MIN_DF, MAX_FEATURES, NGRAM_RANGE, and STOPWORDS from the cell
# above — same vocabulary-building settings, so the comparison between
# models is not confounded by different feature configurations.

# =============================================== YOUR CODE HERE ===============================================
bow_nb = Pipeline([
    ('vec', CountVectorizer(
        lowercase=True,
        stop_words= ,   # Complete with the right variable
        min_df= ,       # Complete with the right variable
        max_features=,  # Complete with the right variable
        ngram_range=,   # Complete with the right variable
    )),
    ('clf', MultinomialNB())
    # MultinomialNB models raw term counts directly (it expects integer-like
    # frequencies, not TF-IDF weights), so we pair it with CountVectorizer
    # here rather than TfidfVectorizer — a natural contrast with the
    # TF-IDF + Logistic Regression model above.
])

# ------------------------------------------------------------
# Fit on the training set, then predict on the held-out test set
# ------------------------------------------------------------
print('\nTraining BoW + Naive Bayes...')
bow_nb.fit(X_train, y_train)
# As before, .fit() runs both steps in sequence: CountVectorizer learns its
# vocabulary from X_train and transforms it into a count matrix, then
# MultinomialNB is trained on those counts against y_train.

y_pred_nb = bow_nb.predict(X_test)
# X_test is transformed using the vocabulary already learned from training
# (never refit), then the trained classifier predicts a label for each row.

print('\nTraining completed.\n')
# y_pred_nb now holds this model's predictions on X_test — comparing it
# against y_pred (from TF-IDF + Logistic Regression) on the same y_test
# will show whether the choice of vectorizer + classifier actually
# matters for this task, or whether both approaches land in a similar place.

In [ ]:
comparison = pd.DataFrame({
    'model': [
        'TFIDF + LogisticRegression',
        'BoW + NaiveBayes',
    ],
    'accuracy': [
        accuracy_score(y_test, y_pred),
        accuracy_score(y_test, y_pred_nb),
    ],
    'macro_f1': [
        precision_recall_fscore_support(y_test, y_pred, average='macro', zero_division=0)[2],
        precision_recall_fscore_support(y_test, y_pred_nb, average='macro', zero_division=0)[2],
    ]
})

display(comparison)

# Save reusable outputs

We save:
- evaluation metrics
- feature-weight tables
- error-analysis tables
- trained pipelines

These artifacts will support later notebooks and the collaborative paper.

In [ ]:
metrics_df.to_csv(TABLES_DIR / 'nb09-metrics.csv', index=False)
weights_df.to_csv(TABLES_DIR / 'nb09-feature_weights.csv', index=False)
errors.to_csv(TABLES_DIR / 'nb09-errors.csv', index=False)
comparison.to_csv(TABLES_DIR / 'nb09-model_comparison.csv', index=False)

print('Saved metrics, feature weights, and error-analysis tables.')

# Reflection

Important questions to consider:

- What kinds of linguistic signals seem most useful for period prediction?
- Are the strongest features conceptually meaningful or merely stylistic?
- Which periods are most difficult to distinguish?
- How might OCR quality or metadata choices affect the model?
- What changes when using embeddings instead of lexical features?

Final methodological reminder:

> Classification models do not discover historical truth. They learn statistical regularities from the representations and labels we provide.

## Conclusion

This notebook walked through a complete supervised classification workflow — not just training a model, but building the methodological scaffolding around it: a leakage-safe train/test split, a 5-fold cross-validation check on how sensitive results are to a single feature-design choice (`min_df`), a TF–IDF + Logistic Regression baseline, and a second Bag-of-Words + Naive Bayes model trained under matched vocabulary settings for a fair comparison.

A few things are worth carrying forward:

- **Performance is a range, not a point.** The cross-validation step showed that both across folds and across `min_df` settings, scores move — sometimes more than a single train/test split would suggest. Any one accuracy or F1 number in isolation should be read with that variability in mind.
- **Evaluation beyond accuracy mattered.** The confusion matrix showed *which* time bins get confused with which — often adjacent periods rather than distant ones — and the per-class feature weights and misclassified examples gave a qualitative sense of *why*, not just *how often*.
- **High classifier performance is not the same as historical insight.** As the top-weighted features and error cases illustrated, a model predicting `time_bin` successfully may be latching onto genuine conceptual shifts, but could just as easily be picking up archaic spelling, OCR noise, or author-specific style. Distinguishing between these requires the kind of qualitative inspection this notebook practiced, not just a higher score.
- **Representation choices are part of the argument.** TF–IDF+LR and BoW+NB didn't just differ in accuracy — they differed in what kinds of signal they could pick up on and how interpretable their weights were. Swapping the vectorizer or classifier is itself a methodological decision, not a neutral technical detail.

The saved tables (`nb09-metrics.csv`, `nb09-feature_weights.csv`, `nb09-errors.csv`, `nb09-model_comparison.csv`) carry these results forward. Notebook 10 picks up a related but distinct thread — training a custom NER model — while the broader question this notebook raised (what a classifier is *actually* learning from a representation) stays relevant through the rest of Part IV.

```mermaid
flowchart TB
    A0["00<br/>Bootcamp"] --> P1

    subgraph P1["Part I — Corpus building and analysis"]
        direction LR
        A1a["01a<br/>Corpus metadata"] --> A1b["01b<br/>Corpus building"] --> A2["02<br/>Preprocessing"] --> A3["03<br/>Distributions + time"] --> A4a["04a<br/>Lexical exploration"] --> A4b["04b<br/>Embedding"]
    end

    subgraph P2["Part II — Linguistic annotations"]
        direction LR
        A5a["05a<br/>spaCy annotation"] --> A5b["05b<br/>Relation extraction"] --> A6a["06a<br/>NER"] --> A6b["06b<br/>Custom NER"]
    end

    subgraph P3["Part III — Representations"]
        direction LR
        A7["07<br/>BoW + TF-IDF"] --> A8a["08a<br/>Embeddings"] --> A8b["08b<br/>Transformers"]
    end

    subgraph P4["Part IV — Models and interpretation"]
        direction LR
        A9["09<br/>Classification"] --> A10["10<br/>Custom NER training"] --> A11["11<br/>Topic modeling"] --> A12["12<br/>Semantic shift"]
    end

    P1 --> P2
    P2 --> P3
    P3 --> P4

    classDef start fill:#f3f0ff,stroke:#6f42c1,stroke-width:1.5px,color:#111;
    classDef prep fill:#eef7ff,stroke:#1f77b4,stroke-width:1.5px,color:#111;
    classDef annot fill:#eefaf0,stroke:#2ca02c,stroke-width:1.5px,color:#111;
    classDef repr fill:#fff7e6,stroke:#ff8c00,stroke-width:1.5px,color:#111;
    classDef model fill:#fff0f0,stroke:#d62728,stroke-width:1.5px,color:#111;

    classDef highlight fill:#fff3b0,stroke:#f5a623,stroke-width:4px,color:#111;

    class A1a,A1b,A2,A3,A4a,A4b prep;
    class A5a,A5b,A6a,A6b annot;
    class A7,A8a,A8b repr;
    class A9,A10,A11,A12 model;

    class A9 highlight;
```